# Training Notebook
Run the cells in order from top to bottom. This notebook loads `train_data.jsonl` and `val_data.jsonl` from the same directory as the notebook, starts supervised fine-tuning, and updates the training and validation loss curves live in Jupyter. The trained model is saved to the parent `Model` directory using the format `<model-size>-<timestamp>`, for example `3B-20260226_224018`.


In [ ]:
%matplotlib inline


In [ ]:
import os
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import torch
import matplotlib.pyplot as plt
from datetime import datetime
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback, TrainerCallback
from transformers import DataCollatorForSeq2Seq
from IPython.display import DisplayHandle


# ===================== Configuration =====================

LEARNING_RATE = 1e-4

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.0
NEFTUNE_NOISE_ALPHA = 0

NUM_EPOCHS = 3
MAX_SEQ_LENGTH = 2560

SEED = 3407
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

BATCH_SIZE = 6
GRAD_ACCUM = 3


# MODEL_NAME = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit"
# MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit"
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"


def resolve_notebook_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Release" / "Training",
    ]
    for candidate in candidates:
        if (candidate / "train_v6_plot.ipynb").exists():
            return candidate.resolve()
    return Path.cwd().resolve()


NOTEBOOK_DIR = resolve_notebook_dir()
MODEL_DIR = NOTEBOOK_DIR.parent / "Model"
TRAIN_FILE = str(NOTEBOOK_DIR / "train_data.jsonl")
VAL_FILE = str(NOTEBOOK_DIR / "val_data.jsonl")
DTYPE = None
LOAD_IN_4BIT = True

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

if "8B" in MODEL_NAME:
    MODEL_SIZE = "8B"
elif "3B" in MODEL_NAME:
    MODEL_SIZE = "3B"
elif "1B" in MODEL_NAME:
    MODEL_SIZE = "1B"
else:
    MODEL_SIZE = "Model"

SAVE_DIR = str(MODEL_DIR / f"{MODEL_SIZE}-{run_id}")
OUTPUT_DIR = str(NOTEBOOK_DIR / f"outputs_{run_id}")
# ======================================================

FIXED_INSTRUCTION = (
    "You are an industrial automation planning system.\n"
    "Given the system configuration (Input JSON) and the history of executed operations [Past Steps], "
    "predict ONLY the exact SINGLE next step required to fulfill the order.\n\n"
    "Format your output exactly as:\n"
    "Step N | Op: ... | Cost: ... | Dur: ...\n\n"
    "If the order is completely fulfilled and no further actions are needed, output:\n"
    "Step N | Op: End | Cost: 0.000 | Dur: 0.000\n\n"
    "Do NOT output any extra keys, JSON, reasoning text, or multiple steps."
)

PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)

RESPONSE_MARKER = "### Response:\n"




In [ ]:
def find_sublist(haystack, needle):
    if not needle:
        return -1
    n = len(needle)
    for i in range(0, len(haystack) - n + 1):
        if haystack[i:i+n] == needle:
            return i
    return -1




In [ ]:
class LiveLossPlotCallback(TrainerCallback):
    def __init__(self):
        self.train_epochs = []
        self.train_losses = []
        self.eval_epochs = []
        self.eval_losses = []
        self.fig = None
        self.ax = None
        self.display_handle = None
        self.total_epochs = None
        self.enable_live = self._is_notebook()

    @staticmethod
    def _is_notebook():
        try:
            from IPython import get_ipython
            shell = get_ipython()
            return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"
        except Exception:
            return False

    def _ensure_figure(self):
        if self.fig is not None:
            return
        plt.style.use("seaborn-v0_8-whitegrid")
        plt.rcParams.update({
            "font.family": "serif",
            "font.size": 12,
            "axes.labelsize": 12,
            "axes.titlesize": 13,
            "legend.fontsize": 10,
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
            "axes.linewidth": 1.0,
        })
        self.fig, self.ax = plt.subplots(figsize=(8.0, 5.0), dpi=150)

    def _draw(self):
        self._ensure_figure()
        self.ax.clear()

        if self.train_epochs:
            self.ax.plot(
                self.train_epochs,
                self.train_losses,
                color="#1f77b4",
                linewidth=2.0,
                marker="o",
                markersize=3.5,
                label="Train Loss",
            )
        if self.eval_epochs:
            self.ax.plot(
                self.eval_epochs,
                self.eval_losses,
                color="#d62728",
                linewidth=2.2,
                marker="s",
                markersize=4.0,
                label="Eval Loss",
            )

        self.ax.set_title("Training and Evaluation Loss")
        self.ax.set_xlabel("Epoch")
        self.ax.set_ylabel("Loss")
        if self.total_epochs is not None:
            self.ax.set_xlim(0, float(self.total_epochs))
        self.ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.5)
        self.ax.legend(loc="best", frameon=True)
        self.fig.tight_layout()

        if self.enable_live:
            if self.display_handle is None:
                self.display_handle = DisplayHandle()
                self.display_handle.display(self.fig)
            else:
                self.display_handle.update(self.fig)

    def on_train_begin(self, args, state, control, **kwargs):
        self.total_epochs = args.num_train_epochs

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        updated = False
        epoch = logs.get("epoch", state.epoch)
        if epoch is None:
            return

        train_loss = logs.get("loss", logs.get("train_loss"))
        if train_loss is not None:
            self.train_epochs.append(float(epoch))
            self.train_losses.append(float(train_loss))
            updated = True

        if "eval_loss" in logs:
            self.eval_epochs.append(float(epoch))
            self.eval_losses.append(float(logs["eval_loss"]))
            updated = True

        if updated:
            self._draw()

    def save_plot_and_data(self, save_dir):
        if not (self.train_epochs or self.eval_epochs):
            return
        os.makedirs(save_dir, exist_ok=True)
        self._draw()

        png_path = os.path.join(save_dir, "loss_curve.png")
        pdf_path = os.path.join(save_dir, "loss_curve.pdf")
        hist_path = os.path.join(save_dir, "loss_history.json")

        self.fig.savefig(png_path, dpi=300, bbox_inches="tight")
        self.fig.savefig(pdf_path, bbox_inches="tight")
        with open(hist_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "train": [{"epoch": e, "loss": l} for e, l in zip(self.train_epochs, self.train_losses)],
                    "eval": [{"epoch": e, "loss": l} for e, l in zip(self.eval_epochs, self.eval_losses)],
                },
                f,
                ensure_ascii=False,
                indent=2,
            )




In [ ]:
def main():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32)
    print("TF32 cudnn :", torch.backends.cudnn.allow_tf32)
    print("bf16 supported:", torch.cuda.is_bf16_supported())

    # ===== 1) Load model/tokenizer =====
    print(f"⬇️ Loading model: {MODEL_NAME} ...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
        token=os.environ.get("HF_TOKEN"),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ===== 2) LoRA =====
    print(f"🔧 LoRA Config: r={LORA_R}, alpha={LORA_ALPHA} ...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth", 
        random_state=SEED,
    )

    marker_ids = tokenizer(RESPONSE_MARKER, add_special_tokens=False).input_ids
    eos = tokenizer.eos_token or ""

    # ===== 3) Load dataset =====
    print("📂 Loading datasets ...")
    ds = load_dataset("json", data_files={"train": TRAIN_FILE, "val": VAL_FILE})
    print(f"✅ train={len(ds['train'])}, val={len(ds['val'])}")

    # ===== 4) Pre-tokenize =====
    def preprocess(examples):
        inputs = examples["input"]
        outputs = examples["output"]

        texts = [
            PROMPT_TEMPLATE.format(
                instruction=FIXED_INSTRUCTION,
                input=inp,
                output=out,
            ) + eos
            for inp, out in zip(inputs, outputs)
        ]

        enc = tokenizer(
            texts,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=False,
            add_special_tokens=False,
        )

        labels = []
        for ids in enc["input_ids"]:
            pos = find_sublist(ids, marker_ids)
            if pos == -1:
                lab = [-100] * len(ids)
            else:
                start = pos + len(marker_ids)
                lab = ids[:]  # copy
                for i in range(start):
                    lab[i] = -100
            labels.append(lab)

        enc["labels"] = labels
        return enc

    train_ds = ds["train"].map(
        preprocess,
        batched=True,
        remove_columns=ds["train"].column_names,
        desc="Tokenizing train",
    )
    val_ds = ds["val"].map(
        preprocess,
        batched=True,
        remove_columns=ds["val"].column_names,
        desc="Tokenizing val",
    )

    # ===== 5) TrainingArguments =====
    args = TrainingArguments(
        output_dir=OUTPUT_DIR,

        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,


        gradient_checkpointing=True, 


        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,

        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),

        optim="adamw_8bit",
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",

        eval_strategy="steps",
        eval_steps=300,
        save_strategy="steps",
        save_steps=600,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        logging_steps=20,
        logging_first_step=True,
        report_to="none",

        seed=SEED,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,
        group_by_length=True,
        remove_unused_columns=False,
    )

    early_stop = EarlyStoppingCallback(
        early_stopping_patience=3,
        early_stopping_threshold=0.001,
    )
    live_loss_plot_callback = LiveLossPlotCallback()

    # Use dynamic padding during batching.
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, 
        padding=True,
        pad_to_multiple_of=8,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=args,
        data_collator=data_collator,
        callbacks=[early_stop, live_loss_plot_callback],
        packing=False,
        neftune_noise_alpha=NEFTUNE_NOISE_ALPHA,
    )

    # ===== 6) Train =====
    print("=" * 60)
    print(f"🎯 TRAINING STARTED | Run ID: {run_id}")
    print(f"📊 Config: Batch={BATCH_SIZE} | GradAccum={GRAD_ACCUM} | Checkpointing=ON")
    print(f"💾 Save Dir: {SAVE_DIR}")
    print("=" * 60)
    
    train_out = trainer.train()
    print(train_out)

    # ===== 7) Save =====
    print(f"💾 Saving model to {SAVE_DIR} ...")
    os.makedirs(SAVE_DIR, exist_ok=True)
    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)

    # Save the run configuration for reference.
    with open(os.path.join(SAVE_DIR, "training_config.json"), "w", encoding="utf-8") as f:
        json.dump({
            "run_id": run_id,
            "model": MODEL_NAME,
            "max_seq_length": MAX_SEQ_LENGTH,
            "lora_r": LORA_R,
            "batch_size": BATCH_SIZE,
            "grad_accum": GRAD_ACCUM,
            "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,
            "lr": LEARNING_RATE,
            "epochs": NUM_EPOCHS,
            "gradient_checkpointing": True
        }, f, ensure_ascii=False, indent=2)

    print("✅ Model Saved Successfully.")

    print("📈 Eval ...")
    ev = trainer.evaluate()
    print(ev)
    live_loss_plot_callback.save_plot_and_data(SAVE_DIR)
    print(f"📉 Loss curve saved to: {os.path.join(SAVE_DIR, 'loss_curve.png')}")




In [ ]:
if __name__ == "__main__":
    main()
